In [ ]:
#Day-1: Production Preprocessing Pipelines & Feature Selection
#Engineering Problem
#Real-world datasets contain missing values, diverse numeric distributions, and redundant or low-variance features. 
#Performing global data transformations prior to splitting leads to target and feature leakage, artificially inflating offline performance metrics.
#Technical Objective
#1.Implement an immutable Scikit-Learn `ColumnTransformer` pipeline that fits strictly on training partitions and transforms unseen holdout data.
#2.Integrate variance thresholding and tree-based Recursive Feature Elimination (RFE) to prune noisy or collinear variables systematically without data leakage.

In [4]:
# Data Ingestion & Leak-Free Splitting
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# 1. Fetch raw data
raw_data = fetch_california_housing(as_frame=True) #No need of download dataset scikit-learn build-in fun loads directly
X = raw_data.data.copy()
y = raw_data.target.copy()

# 2. Inject artificial missingness & synthetic noise to simulate dirty production feeds
np.random.seed(42)
missing_mask = np.random.rand(*X.shape) < 0.05
X = X.mask(missing_mask)

# Add collinear & low-variance noise columns
X["Synthetic_Constant"] = 1.0
X["Synthetic_Collinear_MedInc"] = X["MedInc"] * 1.5 + np.random.normal(0, 0.01, size=len(X))

# 3. STRICT SPLIT: Split before any statistics (mean, median, variance) are computed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Null counts in train:\n{X_train.isnull().sum()[X_train.isnull().sum() > 0]}")

X_train shape: (16512, 10)
X_test shape:  (4128, 10)
Null counts in train:
MedInc                        788
HouseAge                      819
AveRooms                      823
AveBedrms                     822
Population                    801
AveOccup                      791
Latitude                      810
Longitude                     841
Synthetic_Collinear_MedInc    788
dtype: int64


In [2]:
#Preprocessing Pipeline & Feature Selection Assembly
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.ensemble import ExtraTreesRegressor

# Define feature boundaries
numeric_features = X_train.columns.tolist()

# 1. Transform: Imputation + Outlier-robust scaling
preprocessing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())  
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", preprocessing_pipeline, numeric_features)
    ],
    remainder="drop"
)

# 2. Feature Selection: Zero-variance elimination + Feature importance pruning
feature_selector = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=0.0)), 
    ("model_selector", SelectFromModel(
        estimator=ExtraTreesRegressor(n_estimators=50, random_state=42, n_jobs=-1),
        threshold="median" 
    ))
])

# 3. Full Unified Pipeline
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selector", feature_selector)
])

# FIT ONLY ON TRAIN DATA
full_pipeline.fit(X_train, y_train)

# Transform both partitions
X_train_transformed = full_pipeline.transform(X_train)
X_test_transformed = full_pipeline.transform(X_test)

print("Pipeline fit and transformation completed successfully.")
print(f"Transformed Train shape: {X_train_transformed.shape}")
print(f"Transformed Test shape:  {X_test_transformed.shape}")

Pipeline fit and transformation completed successfully.
Transformed Train shape: (16512, 5)
Transformed Test shape:  (4128, 5)


In [7]:
# Diagnostics & Validation Table
# Extract mask of retained features
variance_mask = full_pipeline.named_steps["feature_selector"].named_steps["variance_filter"].get_support()
selected_features_after_variance = [feat for feat, keep in zip(numeric_features, variance_mask) if keep]

model_mask = full_pipeline.named_steps["feature_selector"].named_steps["model_selector"].get_support()
final_selected_features = [feat for feat, keep in zip(selected_features_after_variance, model_mask) if keep]

# Summary diagnostic table
diagnostics_df = pd.DataFrame({
    "Original Feature Count": [X_train.shape[1]],
    "After Variance Filter": [len(selected_features_after_variance)],
    "Final Retained Features": [len(final_selected_features)],
    "Retained Feature Names": [", ".join(final_selected_features)],
    "Test Set NaN Count": [np.isnan(X_test_transformed).sum()]
})

display(diagnostics_df.T.rename(columns={diagnostics_df.index[0]: "Pipeline Audit Metrics"}))

,Pipeline Audit Metrics
Original Feature Count,10
After Variance Filter,9
Final Retained Features,5
Retained Feature Names,"MedInc, AveOccup, Latitude, Longitude, Synthet..."
Test Set NaN Count,0


In [5]:
# Explanation of every cell we performed 
# Cell 2: Getting and Splitting the Data
# Instead of messing with downloaded CSVs, we use a built-in function to grab the California housing dataset directly.
# We then randomly delete some numbers to mimic the messy data you'd see in the real world.
# Right after that, we split the data into a "training" pile and a "testing" pile.
# We do this early so the model can't accidentally peek at the test data and cheat later on.

# Cell 3: Cleaning and Filtering (The Pipeline)
# This cell is all about cleaning up the mess and picking the best data.
# First, we fill in the blanks using the median, and we scale the numbers so they are easier for the model to read—we use "robust" methods for this so extreme outliers (like multi-million dollar mansions) don't screw up our math.
# We package these cleaning steps into one neat pipeline. Then, we prune the bad features.
# We drop useless columns that never change, use a quick algorithm to rank the rest, and throw out the bottom 50% so we're only training on the most useful information.

# Cell 4: Checking Our Work
# Here, we're just looking under the hood to see what survived.
# We pull the final results out of the pipeline and match them up with the original column names so we can clearly read exactly which features made the final cut.

# The `ColumnTransformer` successfully imputed missing values using the training set median and scaled features to a mean of 0 and standard deviation of 1.
# Because `fit()` was only called on `X_train`, there is zero data leakage into `X_test` [cite: 1.1.5].